# Price exploration — data agent and indicators

Working notebook for Achille (Person 2). We look at the data and prototype the indicators here,
then the final version goes into `data/data_agent.py` (`compute_features`).

**Before committing: Clear All Outputs.**

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath(".."))   # repo root, to import data/ and agents/

from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1. Load the prices

`USE_MOCK = True` uses fake data (random walks), handy to test without internet.
For real prices, set `False`: `data/etf_yahoo_map.csv` must exist (run `python data/check_etfs_yfinance.py` once).

In [ ]:
USE_MOCK = False
START = date(2025, 1, 1)

if USE_MOCK:
    from data.mock import mock_prices
    df = mock_prices(days=400)
else:
    from data.data_agent import fetch_prices
    df = fetch_prices(START, date.today())

df.head()

In [ ]:
# One column per ETF, like in Excel
prices = df.pivot(index="date", columns="ticker", values="close_eur")
prices.index = pd.to_datetime(prices.index)
prices.tail()

## 2. Data quality checks

In [ ]:
summary = df.groupby("ticker").agg(
    first_date=("date", "min"),
    last_date=("date", "max"),
    n_days=("date", "size"),
    last_price=("close_eur", "last"),
)
print(f"{df['ticker'].nunique()} ETFs, from {df['date'].min()} to {df['date'].max()}")
summary.sort_values("n_days").head(10)   # ETFs with the shortest history

In [ ]:
# Largest one-day moves: check by hand (real move or data error?)
rets = prices.pct_change(fill_method=None)
extremes = rets.abs().stack().sort_values(ascending=False).head(10)
extremes.rename("abs_move").to_frame()

## 3. Visualise

In [ ]:
# Prices rebased to 100
rebased = prices / prices.bfill().iloc[0] * 100
pick = ["WORLD", "NASDAQ", "GOLD", "EUR GOVT BOND", "BITCOIN"]
rebased[[c for c in pick if c in rebased]].plot(figsize=(12, 5), title="Prices rebased to 100")
plt.ylabel("rebased to 100"); plt.show()

In [ ]:
# Annualised return and volatility per ETF
stats = pd.DataFrame({
    "annual_return": rets.mean() * 252,
    "annual_vol": rets.std() * np.sqrt(252),
})
ax = stats.plot.scatter(x="annual_vol", y="annual_return", figsize=(10, 7))
for t, r in stats.iterrows():
    ax.annotate(t, (r["annual_vol"], r["annual_return"]), fontsize=7)
plt.title("Return and volatility per ETF"); plt.show()

In [ ]:
# Correlations between ETFs: which ones move together?
corr = rets.corr()
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index, fontsize=7)
fig.colorbar(im); plt.title("Correlation of daily returns"); plt.show()

## 4. Indicator prototypes

The chosen strategy: **trend + timing**. We prototype here, on the `prices` table (one column per ETF),
then move the code into `compute_features`.

Rule: an indicator at date *t* only uses prices up to *t* included.

In [ ]:
ind = {}

# --- Trend ---
ind["dist_sma200"] = prices / prices.rolling(200).mean() - 1             # distance to the 200-day average
ind["mom_3m"] = prices.pct_change(63, fill_method=None)                   # 3-month momentum (63 trading days)
ind["mom_12_1m"] = prices.shift(21).pct_change(231, fill_method=None)     # 12 months, skipping the last month

# --- Timing ---
def rsi(p, n):
    change = p.diff()
    up = change.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    down = (-change.clip(upper=0)).ewm(alpha=1/n, adjust=False).mean()
    return 100 - 100 / (1 + up / down)

ind["rsi_2"] = rsi(prices, 2)
ind["zscore_20d"] = (prices - prices.rolling(20).mean()) / prices.rolling(20).std()

# --- Risk ---
ind["vol_20d"] = rets.rolling(20).std() * np.sqrt(252)
ind["drawdown_60d"] = prices / prices.rolling(60).max() - 1

# To do: beta to MSCI World, cross-ETF ranks, market regime

# Quick look at one ETF
etf = "WORLD" if "WORLD" in prices else prices.columns[0]
pd.DataFrame({k: v[etf] for k, v in ind.items()}).dropna(how="all").tail(10)

In [ ]:
# 2-day RSI and z-score over 6 months for one ETF
etf = "NASDAQ" if "NASDAQ" in prices else prices.columns[0]
window = prices.index[-126:]
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
prices.loc[window, etf].plot(ax=axes[0], title=etf)
ind["rsi_2"].loc[window, etf].plot(ax=axes[1], title="2-day RSI"); axes[1].axhline(10, ls="--"); axes[1].axhline(90, ls="--")
ind["zscore_20d"].loc[window, etf].plot(ax=axes[2], title="20-day z-score"); axes[2].axhline(-2, ls="--"); axes[2].axhline(2, ls="--")
plt.tight_layout(); plt.show()

In [ ]:
# Format expected by the pipeline (FEATURES): one row per date, ETF and indicator
features = (pd.concat({k: v.stack() for k, v in ind.items()}, names=["feature", "date", "ticker"])
            .rename("value").reset_index().dropna())
features = features[["date", "ticker", "feature", "value"]]
features.head()